# Music Generator — Google Colab Setup

This notebook sets up the full Music Generator system on Colab with GPU,
including the Magenta microservice (TF1 runs fine on Colab's x86 Linux).

## Step 1: Clone the repo

In [ ]:
# Clone your repo (replace with your actual repo URL)
!git clone https://github.com/YOUR_USERNAME/Music_Generator.git
%cd Music_Generator

## Step 2: Install Magenta (TF1 — works on Colab's x86 Linux)

In [ ]:
# Magenta needs Python 3.7-3.9, Colab currently uses 3.10+
# We install magenta in a conda sub-environment and run it as a subprocess
!pip install -q condacolab
import condacolab
condacolab.install()  # This will restart the runtime — that's expected

In [ ]:
# After runtime restart, run this cell
import condacolab
condacolab.check()

# Create magenta environment
!conda create -n magenta_env python=3.9 -y
!conda run -n magenta_env pip install magenta fastapi uvicorn

## Step 3: Download Magenta models

In [ ]:
!mkdir -p models
!wget -q http://download.magenta.tensorflow.org/models/attention_rnn.mag -P models/
!wget -q http://download.magenta.tensorflow.org/models/polyphony_rnn.mag -P models/
!ls -lh models/

## Step 4: Install main app dependencies

In [ ]:
# Install AutoGen from local source
!pip install -q -e "./autogen/python/packages/autogen-core"
!pip install -q -e "./autogen/python/packages/autogen-agentchat"
!pip install -q -e "./autogen/python/packages/autogen-ext[openai]"

# Install other dependencies
!pip install -q fastapi "uvicorn[standard]" "pydantic>=2.0" pydantic-settings \
    httpx music21 mido python-multipart websockets loguru \
    openai anthropic google-generativeai

## Step 5: Configure API keys

In [ ]:
import os
from google.colab import userdata

# Set your API key — use Colab Secrets (key icon in left sidebar)
# Add a secret named GEMINI_API_KEY with your key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Write .env file
env_content = f"""MG_LLM_BACKEND=gemini
MG_LLM_MODEL=gemini-2.0-flash
MG_LLM_API_KEY={GEMINI_API_KEY}
MG_OLLAMA_BASE_URL=http://localhost:11434
MG_MUSIC_ENGINE=magenta
MG_MAGENTA_URL=http://localhost:8001
MG_HOST=0.0.0.0
MG_PORT=8000
MG_MAX_GROUP_CHAT_MESSAGES=50
MG_CRITIC_PASS_THRESHOLD=0.8
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('✓ .env written')

## Step 6: Start Magenta microservice (background)

In [ ]:
import subprocess, time

magenta_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'magenta_env',
     'uvicorn', 'src.engine.magenta_service:app', '--port', '8001'],
    env={**os.environ,
         'MELODY_MODEL': './models/attention_rnn.mag',
         'POLY_MODEL': './models/polyphony_rnn.mag'},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(5)

# Verify
import requests
try:
    r = requests.get('http://localhost:8001/health')
    print('Magenta service:', r.json())
except:
    print('Magenta service not ready yet — wait a few more seconds and check again')

## Step 7: Start main API server (background)

In [ ]:
main_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(3)

# Verify
try:
    r = requests.get('http://localhost:8000/api/health')
    print('Main service:', r.json())
except:
    print('Main service not ready yet')

## Step 8: Generate music!

In [ ]:
import requests, json

response = requests.post('http://localhost:8000/api/generate', json={
    'request': {
        'description': 'A gentle Chinese-style piano piece with pentatonic melody',
        'genre': 'classical',
        'mood': 'peaceful',
        'tempo': 80,
        'key': 'C',
        'scale_type': 'chinese_pentatonic',
        'sections': ['intro', 'verse', 'chorus', 'outro'],
        'bars_per_section': 4
    }
})

result = response.json()
print(f"Status: {result['status']}")
print(f"Summary: {json.dumps(result['summary'], indent=2)}")
print(f"\nMessages ({len(result['messages'])}):")
for msg in result['messages']:
    preview = msg['content'][:150]
    print(f"  [{msg['source']}] {preview}...")

## Step 9: Play the MIDI file

In [ ]:
import glob
from IPython.display import Audio

# Find generated MIDI files
midi_files = sorted(glob.glob('output/*.mid'), key=os.path.getmtime, reverse=True)

if midi_files:
    print(f'Found MIDI: {midi_files[0]}')

    # Convert MIDI to WAV for playback using fluidsynth
    !apt-get install -y -q fluidsynth
    !wget -q https://github.com/FluidSynth/fluidsynth/raw/master/sf2/VintageDreamsWaves-v2.sf2 -O soundfont.sf2 2>/dev/null || true

    # Try with a General MIDI soundfont
    !apt-get install -y -q fluid-soundfont-gm 2>/dev/null
    SF2 = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

    midi_path = midi_files[0]
    wav_path = midi_path.replace('.mid', '.wav')
    !fluidsynth -ni "{SF2}" "{midi_path}" -F "{wav_path}" -r 44100

    display(Audio(wav_path))
else:
    print('No MIDI files generated yet.')
    # Check Magenta drafts
    draft_files = glob.glob('output/drafts/*.mid')
    if draft_files:
        print(f'Found Magenta drafts: {draft_files}')

## Step 10: Download the MIDI

In [ ]:
from google.colab import files

if midi_files:
    files.download(midi_files[0])
    print(f'Download: {midi_files[0]}')

## Cleanup

In [ ]:
# Stop background services when done
magenta_proc.terminate()
main_proc.terminate()
print('Services stopped.')